In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 1.4 Kepler Orbits and the Two-Body Problem

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume I — Elementary Mechanics",
    number="1.4",
    title="Kepler Orbits and the Two-Body Problem",
    blurb="Conservation laws as integrator checks, the closed inverse-square "
    "ellipse, Kepler's third law, and why a slightly different force law makes "
    "orbits precess.",
    difficulty="intermediate",
    estimate="60–90 min",
)

## Notebook overview

Two bodies pulled together by gravity make the cleanest nontrivial problem in
mechanics: the motion separates into a trivial drift of the centre of mass and
a single *effective* particle orbiting a fixed centre under a central force.
That reduction is what lets a one-line force law reproduce Kepler's three
laws. We will integrate the orbit numerically and lean on the two things the
integrator cannot fake (**energy** and **angular momentum**) to certify that
the trajectory is real before we trust anything we measure from it.

Then we ask the question that makes the inverse-square law special. Bertrand's
theorem says that of all power-law central forces, only $1/r^2$ (and the
harmonic $\propto r$) produces orbits that *close* on themselves; every other
exponent traces a precessing rosette. We will *see* that: a perfectly closed
ellipse for $\beta = 2$, and a slowly rotating rosette the moment we nudge the
exponent to $\beta = 2.1$.

We will (1) implement the equation of motion, (2) integrate a bound orbit and
plot it, (3, 4) use energy and angular-momentum conservation as integrator
checks, (5) animate the closed inverse-square ellipse and confirm it closes,
(6) verify Kepler's third law, (7) animate a precessing orbit under a
modified force law, and (8) identify the conserved Laplace–Runge–Lenz vector
that marks the closure.

> **How to read the checks.** Each exercise ends with a validation that
> compares our result to an expected physical fact. A ✗ does *not* by itself
> mean the answer is wrong: it means the output didn't match what the check
> expected, which may be a real error, a different-but-valid convention (a
> sign, a unit, an array order), or simply too tight a tolerance. Treat a ✗ as
> a prompt to locate the discrepancy; passing is strong evidence of
> correctness, not proof.

> **Scope.** This is a working review, not a textbook chapter. For the
> central-force reduction and Bertrand's theorem see Nolting, *Theoretische
> Physik 1* {cite}`nolting1`, and Goldstein, Poole & Safko, *Classical
> Mechanics* {cite}`goldstein`.

## Theory in brief

### Reduction to a one-body central-force problem

Two masses interacting only through their mutual gravity have a Lagrangian that
separates, in centre-of-mass and relative coordinates, into the free motion of
the total mass and the motion of a single fictitious particle of **reduced
mass** $\mu = m_1 m_2 / (m_1 + m_2)$ at the relative separation
$\mathbf r = \mathbf r_1 - \mathbf r_2$. All the physics of the orbit lives in
that one relative coordinate. For clean numerics we set $GM = 1$ and
$\mu = 1$; lengths and times are then in units fixed by that choice, and every
result below is dimensionless.

### Equation of motion

Newtonian gravity gives a central acceleration directed along $-\hat{\mathbf
r}$ with magnitude $1/r^2$. We will generalise the exponent to a tunable
$\beta$ (with $\beta = 2$ the Newtonian case) so we can later break the
inverse-square law on purpose:

```{math}
:label: eq-rhs
\ddot{\mathbf r} = -\frac{1}{r^{\beta}}\,\hat{\mathbf r}
  = -\frac{\mathbf r}{r^{\beta+1}}, \qquad r = |\mathbf r|.
```

We integrate this in the plane with the state $\mathbf s = (x, y, v_x, v_y)$.

### Two constants of motion

A central force does no work tangentially and exerts no torque about the
centre, so the **energy** and the **angular momentum** are conserved. With
$-\mathrm{d}V/\mathrm{d}r$ equal to the radial force $-1/r^{\beta}$, the
potential is $V(r) = r^{1-\beta}/(1-\beta)$, which is the familiar $V = -1/r$
when $\beta = 2$. The total energy is

```{math}
:label: eq-energy
E = \tfrac12 v^2 + V(r), \qquad V(r) = \frac{r^{1-\beta}}{1-\beta}
  \;\xrightarrow{\ \beta=2\ }\; -\frac1r .
```

In two dimensions the angular momentum is the scalar

```{math}
:label: eq-Lz
L_z = x\,v_y - y\,v_x .
```

Both are *independent* of the integrator: a correct trajectory holds them
fixed, so their drift is our most honest measure of numerical error.

### Bound orbits, vis-viva, and Kepler's third law

For the inverse-square law ($\beta = 2$), $E < 0$ is a bound elliptical orbit
and $E > 0$ an unbound hyperbola. The **vis-viva** relation ties the energy to
the semi-major axis $a$,

```{math}
:label: eq-visviva
E = -\frac{1}{2a} \quad\Longleftrightarrow\quad a = -\frac{1}{2E},
```

and **Kepler's third law** fixes the orbital period from $a$ alone:

```{math}
:label: eq-kepler3
T = 2\pi\sqrt{a^{3}} \qquad (GM = 1).
```

We take both as given here; Goldstein {cite}`goldstein`, Ch. 3, carries the
derivations out in full from the orbit equation.

### Why the ellipse closes — Bertrand's theorem

That a bound orbit returns *exactly* to its starting point after one radial
oscillation is special, not generic. **Bertrand's theorem** states that among
all power-law central forces, only the inverse-square ($\beta = 2$) and the
harmonic ($F \propto r$) laws give orbits that close. For any other exponent
the orbit advances by a nonzero angle each radial period and slowly fills an
annulus: a precessing rosette. Exercise 7 makes this concrete by setting
$\beta = 2.1$. See Goldstein {cite}`goldstein` for the proof.

### The orbit geometry

A bound Kepler orbit is an ellipse with the gravitating centre at one **focus**
(not the centre). The orbiting particle's position is the radius vector
$\mathbf r$ from that focus; the orbit's size is set by the semi-major axis
$a$, and the closest approach (the **perihelion**) lies along the major axis.
These are the quantities the conservation laws and Kepler's third law
{eq}`eq-kepler3` are written in.

In [ ]:
# (solution hidden on the public site)


---
## Setup

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

from ecp import draw, validate
from ecp.animate import show

rng = np.random.default_rng(0)

GM = 1.0  # gravitational parameter (units choice); reduced mass mu = 1
# A bound elliptical initial condition: r = 1, purely tangential speed 1.2,
# which exceeds the circular speed 1.0, so this point is the perihelion.
S0_BOUND = np.array([1.0, 0.0, 0.0, 1.2])


def energy(s, beta=2.0):
    """Total specific energy E = v^2/2 + V(r) of an orbit.

    Conserved along the motion; its drift gauges the integrator. Works on one state or a history.

    Parameters
    ----------
    s : array_like
        State ``[x, y, vx, vy]`` or its time history.
    beta : float, optional
        Force-law exponent (default 2).

    Returns
    -------
    float or numpy.ndarray
        The specific energy.
    """
    x, y, vx, vy = s
    r = np.hypot(x, y)
    return 0.5 * (vx**2 + vy**2) + r ** (1.0 - beta) / (1.0 - beta)


def Lz(s):
    """Specific angular momentum L_z = x·vy − y·vx.

    Conserved for any central force; its constancy is a second integrator check.

    Parameters
    ----------
    s : array_like
        State or state history.

    Returns
    -------
    float or numpy.ndarray
        The angular momentum.
    """
    x, y, vx, vy = s
    return x * vy - y * vx


def perihelion_event(t, s, beta):
    """Event: zero when the radial velocity vanishes (selects minima of r)."""
    x, y, vx, vy = s
    return x * vx + y * vy  # = r * dr/dt


perihelion_event.direction = 1.0  # r increasing through zero -> perihelion

## Exercise 1 — Implement the equation of motion

In the theory section we wrote the central-force law, {eq}`eq-rhs`: the
acceleration points straight back toward the centre, anti-parallel to
$\hat{\mathbf r}$, with magnitude $1/r^{\beta}$. Everything else in the
notebook (the orbit, the conserved quantities, the period) follows from
integrating this one vector equation. This exercise turns {eq}`eq-rhs` into
code and checks it at a test point before we trust it downstream.

1. Using {eq}`eq-rhs`, implement `rhs(t, s, beta=2.0)` for the state
   $\mathbf s = (x, y, v_x, v_y)$, returning
   $(\dot x, \dot y, \dot v_x, \dot v_y)$ (`numpy.hypot` for $r$).
2. Evaluate the acceleration at a test state and confirm it equals
   $-\hat{\mathbf r}/r^{2}$ for the Newtonian case $\beta = 2$ — the validation
   below does exactly this.

In [ ]:
# (solution hidden on the public site)


### Validation 1 — the acceleration is $-GM/r^2\,\hat{\mathbf r}$

At a test position the acceleration returned by `rhs` (its last two components)
must point anti-parallel to the radial unit vector with magnitude $1/r^2$. If
this ✗-es, look first at the sign and at the power of $r$ in `rhs`.

In [ ]:
s_test = np.array([0.6, 0.8, 0.1, -0.2])  # r = 1 here, so the check reads cleanly
r_test = np.hypot(s_test[0], s_test[1])
rhat = s_test[:2] / r_test
a_vec = np.array(rhs(0.0, s_test, 2.0)[2:])
validate.close(a_vec, -rhat / r_test**2, "acceleration is -GM/r² r̂ (β=2)", rtol=1e-10)

## Exercise 2 — Integrate a bound orbit and plot it

With {eq}`eq-rhs` in hand, the orbit is fixed entirely by the initial position
and velocity. We launch from `S0_BOUND`: radius 1 with a purely tangential
speed of 1.2, faster than the circular speed 1.0, so the orbit is a bound
ellipse with the force centre at one focus. This exercise integrates that orbit
and draws it.

1. Integrate {eq}`eq-rhs` from `S0_BOUND` over one period with
   `scipy.integrate.solve_ivp` (the `DOP853` high-order integrator), sampling on
   a dense `t_eval` grid so the curve is smooth rather than a chain of straight
   segments.
2. Plot $y$ vs $x$ with equal aspect, mark the focus at the origin, and confirm
   by eye that the ellipse closes.

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — Energy as an integrator check

The energy {eq}`eq-energy` is conserved by the exact dynamics, so any drift in
the *computed* energy along the trajectory is pure numerical error. This is the
canonical "did I integrate it correctly?" test. This exercise evaluates
{eq}`eq-energy` along the orbit from Exercise 2 and plots its relative drift.

1. Using {eq}`eq-energy`, compute $E(t)$ along the trajectory (the `energy`
   helper).
2. Plot the relative drift $(E - E_0)/|E_0|$ and confirm it stays at the level
   of the solver tolerance — the validation makes this quantitative.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.conserved(E_t, "total energy conserved (integrator check)", rel_drift=1e-6)

## Exercise 4 — Angular-momentum conservation

A central force exerts no torque about the centre, so the angular momentum
{eq}`eq-Lz` is the orbit's second constant of motion, and the geometric
content of Kepler's *second* law: the radius vector sweeps equal areas in equal
times, because the areal rate is exactly $L_z/2$. This exercise checks that
$L_z$ is held fixed along the integrated orbit.

1. Using {eq}`eq-Lz`, compute $L_z(t)$ along the trajectory (the `Lz` helper)
   and plot its relative drift.
2. As a visual of Kepler's second law, shade the area swept over one short early
   interval and one short late interval and note they are equal for equal
   elapsed time.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.conserved(L_t, "angular momentum L_z conserved", rel_drift=1e-6)

```{admonition} With your assistant
:class: tip
Ask your assistant for orbit-integration code for this potential, written any
way it likes — then hold it to the two gates this notebook just built: energy
flat to Exercise 3's tolerance, angular momentum flat to Exercise 4's. A
physics-flavored check outranks any code review: whoever wrote the code, the
conservation laws are not negotiable. The check is yours.
```

## Exercise 5 — The inverse-square orbit closes (worked animation)

Bertrand's theorem (theory section) promises that the inverse-square ellipse
closes exactly: after one radial period the orbit returns to the same point
moving the same way, so it retraces a single curve forever. This exercise makes
that visible and then *measures* it.

This is the **worked** animation of the notebook: study how it is built. It
animates the body tracing the ellipse with a trailing path over two periods;
the trail lands exactly on top of itself, which is what "closed" means. The
validation does not look at the animation at all: it measures the **apsidal
advance**: the change in polar angle between successive perihelia, minus
$2\pi$. For a closed orbit that advance is zero.

In [ ]:
# (solution hidden on the public site)


### Validation 5 — the orbit closes (zero apsidal advance)

We integrate several radial periods, locate each perihelion as a minimum of
$r$ (a zero of $\dot r$ with $r$ increasing), unwrap the polar angle, and
measure the angle gained between consecutive perihelia. Subtracting $2\pi$
gives the apsidal advance, which must be ≈ 0 for the inverse-square law. A ✗
here points at the perihelion detection or the angle unwrapping, not at any
drawing code.

In [ ]:
def apsidal_advance_deg(beta, s0=S0_BOUND, n_orbits=6, n=20000):
    """Mean perihelion-to-perihelion angle advance, minus 2π, in degrees.

    Nonzero advance is orbital precession; it vanishes for the closed
    inverse-square orbit.

    Parameters
    ----------
    beta : float
        Force-law exponent.
    s0 : array_like, optional
        Initial state.
    n_orbits : int, optional
        Orbits to average.
    n : int, optional
        Samples.

    Returns
    -------
    float
        The apsidal advance per orbit, in degrees.
    """
    t_end = (n_orbits + 0.5) * T0
    t_eval = np.linspace(0.0, t_end, n)
    sol = solve_ivp(
        rhs,
        (0.0, t_end),
        s0,
        t_eval=t_eval,
        method="DOP853",
        rtol=1e-11,
        atol=1e-12,
        args=(beta,),
        events=perihelion_event,
    )
    phi = np.unwrap(np.arctan2(sol.y[1], sol.y[0]))
    phi_peri = np.interp(sol.t_events[0], sol.t, phi)
    return np.degrees(np.mean(np.diff(phi_peri)) - 2.0 * np.pi)


advance_closed = apsidal_advance_deg(2.0)
validate.close(
    advance_closed,
    0.0,
    "inverse-square orbit closes (zero apsidal advance)",
    atol=0.1,
)

## Exercise 6 — Kepler's third law

Kepler's third law {eq}`eq-kepler3` says the period depends only on the
semi-major axis: $T = 2\pi\sqrt{a^3}$ in our units. We can get $a$ without ever
measuring the orbit's geometry (the vis-viva relation {eq}`eq-visviva` gives
it straight from the energy), and we can measure $T$ directly as the time
between perihelion passages. This exercise compares the two.

1. From {eq}`eq-energy` and {eq}`eq-visviva`, compute the semi-major axis
   $a = -1/(2E)$ of the reference orbit.
2. Measure the radial period as the mean perihelion-to-perihelion time (the
   `perihelion_event` located by `solve_ivp`) and compare it to {eq}`eq-kepler3`.
3. Repeat for a few launch speeds and confirm $T^2$ is linear in $a^3$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    T_measured,
    2.0 * np.pi * np.sqrt(a_measured**3),
    "period obeys Kepler's third law",
    rtol=1e-3,
)

## Exercise 7 — Precession under a modified force law (student animation)

Bertrand's theorem cuts the other way too: change the exponent away from
$\beta = 2$ and the orbit *cannot* close: it precesses. We set $\beta = 2.1$,
a small change to {eq}`eq-rhs`, and the ellipse becomes a slowly rotating
rosette. This is the **student-implemented** animation: the setup and the data
are given; you write the player.

1. Integrate {eq}`eq-rhs` with $\beta = 2.1$ from `S0_BOUND` over several
   periods using `scipy.integrate.solve_ivp` (`DOP853`) on a dense `t_eval`, and
   store the trajectory `(x, y)`.
2. Build a `FuncAnimation` of the body tracing the rosette with a trailing path
   — there are many valid ways to do this — then `plt.close(fig)` and end the
   cell with `ecp.animate.show(anim)`.

The validation below checks the **physics of the animated orbit**, not the
drawing code: it measures the apsidal advance and confirms it is clearly
nonzero (the orbit precesses). A ✗ means "re-check the β=2.1 trajectory or the
apsidal measurement," never "the animation is wrong."

In [ ]:
# (solution hidden on the public site)


### Validation 7 — the modified force law precesses

Same apsidal-advance measurement as Exercise 5, now for $\beta = 2.1$. We
expect a clearly nonzero advance per radial period (it comes out near
$+21^\circ$), the quantitative signature of a non-closing orbit.

In [ ]:
advance_precess = apsidal_advance_deg(BETA_PRECESS)
print(f"apsidal advance per radial period ≈ {advance_precess:.2f}°")
validate.check(
    abs(advance_precess) > 1.0,
    "modified force law makes the orbit precess",
    detail=f"apsidal advance = {advance_precess:.2f}° per radial period",
)

## Exercise 8 — The orbit that closes: the Laplace–Runge–Lenz vector

Exercise 5 showed the inverse-square orbit retraces a single closed ellipse, and Exercise 7
showed that the smallest change to the force law sets it precessing. What conserved quantity
marks the difference? For the $1/r^2$ force, and *only* for it, there is an extra conserved
vector beyond energy and angular momentum: the **Laplace–Runge–Lenz vector**

$$\mathbf A=\mathbf v\times\mathbf L-k\,\hat{\mathbf r},$$

which points along the major axis toward perihelion. Its constancy *is* the closure of the
orbit: a fixed $\mathbf A$ means the perihelion never moves. This is the clean instrument for
the whole question — it reads the closure off directly from the conserved vector, with no need
to hunt for minima of $r$ — and it is the deepest fact about the Kepler problem. The smallest
departure from $1/r^2$ destroys this conservation, and the orbit begins to turn.

With the same reference orbit (`S0_BOUND`, $GM=1$):

1. Implement $\mathbf A=\mathbf v\times\mathbf L-k\,\hat{\mathbf r}$ as the
   `lrl_vector` helper and evaluate it along a trajectory integrated over thirty
   radial periods.
2. Confirm its *direction* is fixed to within numerical noise
   (`numpy.arctan2` of its components) — the orbit is exactly closed, no
   precession.
3. Confirm its *length* is $|\mathbf A|/k=e$, the eccentricity.

In [ ]:
# (solution hidden on the public site)


### Validation 8 — the LRL vector is conserved (the orbit is exactly closed)

In [ ]:
validate.close(
    direction_spread,
    0.0,
    "the Laplace–Runge–Lenz vector is conserved — the Kepler orbit is exactly closed, with no precession",
    atol=1e-3,
)
validate.close(
    A_mag.mean() / GM,
    e_orbit,
    "the LRL vector's length encodes the eccentricity, |A|/k = e",
    rtol=1e-3,
)

In [ ]:
# (solution hidden on the public site)


**A thread to follow.** This perfect closure is not generic; it is a property of the
inverse-square force (among the falloff laws). Real planetary orbits precess slightly, and we
now have, in our own code, everything needed to see why. [§2.4](../02-classical-mechanics/central-force.ipynb) shows that the smallest
departure from $1/r^2$ sets the perihelion turning, at a rate we can predict and measure; and
the general-relativity capstone ([§4.8](../04-special-relativity/gr-capstone.ipynb)) shows that Mercury's leftover precession — the part no
Newtonian gravity explains — is one of relativity's first triumphs. The puzzle is one the
reader uncovers; relativity supplies the missing piece.

## Notebook summary

- The inverse-square two-body problem ($\ddot{\mathbf r}=-GM\hat{\mathbf r}/r^2$) integrated
  to a closed bound orbit, with total energy and the angular momentum $L_z$ conserved as
  integrator checks.
- Kepler's third law $T^2\propto a^3$ recovered across orbits, the closure of the inverse-square
  ellipse, and the precession that appears once the force law is modified away from $1/r^2$.
- The **Laplace–Runge–Lenz vector** $\mathbf A=\mathbf v\times\mathbf L-k\,\hat{\mathbf r}$
  conserved for the $1/r^2$ force (direction fixed to $<10^{-3}\,$° over thirty orbits, length
  $|\mathbf A|/k=e$): the exact statement that the orbit closes, and the opening of a thread —
  the orbit closes here, precesses in [§2.4](../02-classical-mechanics/central-force.ipynb), and
  yields Mercury's relativistic $43''$/century in [§4.8](../04-special-relativity/gr-capstone.ipynb).

## Outlook

- Map the apsidal advance as a function of $\beta$ across, say, $1.8$–$2.2$ and
  watch it pass through zero exactly at the inverse-square law: a numerical
  reading of Bertrand's theorem.
- Add a small relativistic $1/r^3$ correction to the force and tune it to
  reproduce a Mercury-like perihelion precession.
- Push the launch speed past escape ($v_0 > \sqrt{2}$ at $r=1$): the orbit
  becomes an unbound hyperbolic flyby. Measure the deflection angle versus
  impact parameter: the bridge to central-force scattering in Volume II.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()